In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

use september
use hourly, with start 2025-09-01  01:00:00
convert to wintertime
convert from l/h to m^3/s 
monthly estimate of concentration = 55 000 000 MPN/100ml

In [2]:
filename = Path(r"C:/Users/karoa/MOHID_internship/MOHID_model_workflow/data/preprocessing/Caudales_PSJ_2025.xlsx")
output_folder = filename.parent

df = pd.read_excel(
    filename,
    sheet_name='Septiembre',
    usecols=['Fecha', 'Caudal salida general (l/h)']
)


In [3]:
df['datetime'] = pd.to_datetime(df['Fecha']) - pd.Timedelta(hours=1) # converting to wintertime
df['flow (m3/s)'] = df['Caudal salida general (l/h)'] / (3600000)
# --- Build the SECONDS column relative to the first timestamp ---
t0 = df['datetime'].iloc[0]
df['seconds'] = (df['datetime'] - t0).dt.total_seconds().astype(int)

# ---  Write the output file ---
# build fileout name
start = df['datetime'].iloc[0]
end = df['datetime'].iloc[-1]

fileout = output_folder / f"discharge_{start.year}_{start.month}_{start.day}_{end.year}_{end.month}_{end.day}.dat"
#fileout = f'WIND_{start.year}_{start.month}_{start.day}_{end.year}_{end.month}_{end.day}.dat'

initial = t0.strftime('%Y. %m. %d. %H. %M. %S.').replace(' 0', ' ')  
# Safer: build the header explicitly from the datetime parts
initial = f"{t0.year}. {t0.month:>2}. {t0.day:>2}. {t0.hour}. {t0.minute}. {t0.second}."


In [5]:
with open(fileout, 'w') as f:
    f.write("TIME_UNITS                : SECONDS\n")
    f.write(f"SERIE_INITIAL_DATA        : {initial}\n")
    f.write("\n")
    f.write("SECONDS                   flow (m3/s)\n")
    f.write("<BeginTimeSerie>\n")
    for _, row in df.iterrows():
        # Skip rows where either component is NaN (from the -9999.9 fill values)
        if pd.isna(row['flow (m3/s)']):
            continue
        f.write(f"{row['seconds']}                           "
                f"{row['flow (m3/s)']:.10g}\n")
    f.write("<EndTimeSerie>\n")